# Daemon (030) Demo

Demonstrate the blocking Unix socket daemon basics, including agent metadata and a simple IPC request.


In [1]:
%load_ext autoreload
%autoreload 2
import json
import os
import socket
import tempfile
from pathlib import Path

from ciphercache.daemon.server import UnixSocketServer
from ciphercache.daemon.state import DaemonConfig, DaemonState
from ciphercache.ipc.framing import decode_single_frame, encode_message


In [2]:
data_dir = Path(tempfile.mkdtemp(prefix="ciphercache-daemon-demo-"))
config = DaemonConfig(data_dir=data_dir, write_agent_metadata=True)
state = DaemonState(config=config)
server = UnixSocketServer(config=config, state=state)
server.setup()
agent_path = data_dir / "agent.json"
json.loads(agent_path.read_text(encoding="utf-8"))


{'data_dir': '/var/folders/0t/w9l_5c597rdglh2kbffq18sr0000gn/T/ciphercache-daemon-demo-c7a0pn3z',
 'gid': 20,
 'pid': 76261,
 'socket_path': '/var/folders/0t/w9l_5c597rdglh2kbffq18sr0000gn/T/ciphercache-daemon-demo-c7a0pn3z/ciphercached.sock',
 'started_at': '2026-02-04T19:05:43+00:00',
 'uid': 501,
 'version': 'v0'}

In [4]:
client, server_sock = socket.socketpair()
request = {"version": "v0", "id": "ping", "type": "request", "op": "ping", "payload": {}}
em = encode_message(request)
print("EM:", em)
client.sendall(encode_message(request))
server._handle_connection(server_sock)
response = decode_single_frame(client.recv(4096))
print(response)
print(server_sock)
client.close()
server_sock.close()


EM: b'\x00\x00\x00F{"version":"v0","id":"ping","type":"request","op":"ping","payload":{}}'
{'version': 'v0', 'id': 'ping', 'type': 'response', 'op': 'ping', 'payload': {'ok': True}}
<socket.socket fd=92, family=1, type=1, proto=0>


In [5]:
server.close()
os.path.exists(config.socket_path)


False

In [6]:
server._handle_signal(15, None)
server.listener.fileno()


AttributeError: 'NoneType' object has no attribute 'fileno'